In [ ]:
# ==============================================================================
# 1 - HIGH-LEVEL OVERVIEW
# 
# The Pipeline Phase: 
# We are entering the Exploratory Data Analysis (EDA) & Data Preprocessing stage 
# of our machine learning pipeline.
# 
# The Problematics: 
# Real-world datasets often load with unoptimized or incorrect data types (e.g., 
# dates read as generic text strings, integers loaded as floats). If left unaddressed, 
# text-based dates prevent us from performing chronological calculations, and heavy, 
# default data types consume excessive memory, slowing down down-stream model training.
# 
# Our Solution: 
# In this step, we establish a robust computational baseline. We import our core 
# analytical libraries, ingest the prepared dataset, explicitly cast the raw arrival 
# timeline into a true datetime schema, and run a structural diagnostic check to audit 
# the data types and row volume.
# ==============================================================================

# ==============================================================================
# 2 - TECHNICAL CODE EXECUTION & BREAKDOWN
# ==============================================================================

import pandas as pd
import numpy as np
import holidays

# Load the project dataset into a pandas DataFrame structure
df = pd.read_csv('ml_ready_bookings.csv')

# Explicitly cast the arrival date column from a generic object/string 
# datatype into a specialized datetime64 object to enable time-series operations
df['arrival_date'] = pd.to_datetime(df['arrival_date'])

# In order to run a first check on the dataset, in particular on the data types, 
# we printed the info about the dataset alongside the total row volume
print(f"Total rows: {df.shape[0]}")
print(df.info())

#### Code Logic & Architecture Explanations:
* **`import holidays`**: Brings in the specialized holiday tracking library, which we will use to map country-specific statutory calendars directly to our checkout timelines without relying on fragile, manual date-mapping tables.
* **`pd.to_datetime()`**: Converts the string representation of dates (e.g., `"2026-07-16"`) into a datetime object. This unlocks access to pandas attributes like `.dt.year`, `.dt.dayofweek`, and seamless chronological filtering.
* **`df.shape[0]`**: Programmatically counts the number of rows along the vertical axis (axis 0) of the DataFrame, giving us an instant confirmation of the data volume we are manipulating.
* **`df.info()`**: Prints a comprehensive summary of the DataFrame, including index data types, non-null counts per column, and memory usage. This is our primary diagnostic tool to check for missing values and optimize memory allocation before feeding data into a machine learning model.

In [ ]:
# ==============================================================================
# 1 - HIGH-LEVEL OVERVIEW
# 
# The Pipeline Phase: 
# Memory Optimization & Type Invariance Enforcement.
# 
# The Problematics: 
# By default, Pandas reads numeric columns containing missing values (NaNs) as heavy 
# float64 values, which unnecessarily inflates the memory footprint of ID fields like 
# 'agent' and 'company'. Furthermore, monetary metrics like Average Daily Rate (ADR) 
# do not require double-precision float64 storage, wasting system RAM during large-scale 
# matrix operations.
# 
# Our Solution: 
# We align the rest of our time-series schemas, convert identifier columns into Pandas' 
# native nullable Integer format ('Int64') to elegantly handle missing records without 
# altering data types, and downcast monetary columns to single-precision float32 to reduce 
# RAM strain.
# ==============================================================================

# ==============================================================================
# 2 - TECHNICAL CODE EXECUTION & BREAKDOWN
# ==============================================================================

# 1. Convert booking_date to datetime to match the arrival timeline schema
df['booking_date'] = pd.to_datetime(df['booking_date'])

# 2. Convert ID columns to Nullable Integers (capital 'I' Int64) to allow NaNs 
# without forcing the entire column to become a float datatype
df['agent'] = df['agent'].astype('Int64')
df['company'] = df['company'].astype('Int64')

# 3. Downcast monetary columns to single-precision float32 to optimize memory efficiency
df['adr'] = df['adr'].astype('float32')
df['total_hotel_revenue_on_arrival_date'] = df['total_hotel_revenue_on_arrival_date'].astype('float32')

# Print structural info again to audit the optimized changes and verify memory reduction
print(df.info())

#### Code Logic & Architecture Explanations:
* **`.astype('Int64')`**: Notice the capital **I**. Unlike standard NumPy integers, Pandas' native `Int64` supports missing values (`NaN`). This prevents an ID column with blank cells from being forcefully cast into a float, preserving the integer property of hotel identifiers.
* **`.astype('float32')`**: Downcasts values from the standard 64-bit float to a 32-bit float. Since the numbers in revenue columns don't require scientific-level precision, cutting the bit-allocation in half reduces the dataset's memory footprint dramatically without losing financial accuracy.

In [ ]:
# ==============================================================================
# 1 - HIGH-LEVEL OVERVIEW
# 
# The Pipeline Phase: 
# Advanced Feature Engineering: Temporal Calendar Mapping.
# 
# The Problematics: 
# Calendar seasonality alone (months, weekdays) cannot capture anomalous booking patterns 
# caused by statutory public holidays. Furthermore, holidays that fall mid-week do not 
# influence leisure cancellations the same way as holidays that create "long weekends" 
# or operational "bridges" (e.g., a holiday falling on a Thursday or Tuesday leading to 
# a taken bridge day).
# 
# Our Solution: 
# We programmatically construct a dynamic Portuguese holiday calendar spanning our entire 
# dataset timeline. We then extract two new core features: a binary holiday indicator 
# (`is_holiday`) and a strict conditional bridge feature (`is_long_weekend`), which explicitly 
# flags holidays falling on Monday, Tuesday, Thursday, or Friday to isolate high-risk 
# leisure travel behavior.
# ==============================================================================

# ==============================================================================
# 2 - TECHNICAL CODE EXECUTION & BREAKDOWN
# ==============================================================================

# 1. Initialize Portuguese Holiday Calendar dynamically based on our dataset's min/max years
pt_holidays = holidays.Portugal(years=range(df['arrival_date'].dt.year.min(), df['arrival_date'].dt.year.max() + 1))

# 2. Engineer Holiday Indicator Features
df['is_holiday'] = df['arrival_date'].apply(lambda d: 1 if d in pt_holidays else 0)
df['holiday_name'] = df['arrival_date'].apply(lambda d: pt_holidays.get(d) if d in pt_holidays else "None")

# 3. Engineer a corrected Long Weekend / Bridge Feature
# A holiday creates a long weekend or an actionable bridge only if it falls on:
# Monday (0), Tuesday (1), Thursday (3), or Friday (4). Saturdays (5) and Sundays (6) are excluded.
df['is_long_weekend'] = np.where(
    (df['is_holiday'] == 1) &
    (df['arrival_date'].dt.dayofweek.isin([0, 1, 3, 4])),
    1,
    0
)

# 4. Print Data Audit and Engineered Feature Summary
print("--- ENVIRONMENT & DATA AUDIT ---")
print(f"Total bookings loaded: {df.shape[0]:,}")
print(f"Memory footprint: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print("\n--- NEW HOLIDAY FEATURES SNEAK PEEK ---")
print(df[df['is_holiday'] == 1][['arrival_date', 'holiday_name', 'is_long_weekend']].drop_duplicates().head(5))

#### Code Logic & Architecture Explanations:
* **`holidays.Portugal()`**: Generates a dynamic Python dictionary where keys are dates and values are official statutory holidays in Portugal. This allows us to scale our timeline seamlessly without manual updates.
* **`df['arrival_date'].dt.dayofweek.isin([0, 1, 3, 4])`**: Evaluates whether the holiday falls on an index representing Monday (0), Tuesday (1), Thursday (3), or Friday (4). This explicitly isolates the "bridge" and "extended weekend" dates where consumer behavior varies most dramatically from typical weeks.
* **`np.where()`**: Implements a vectorized ternary evaluation (`IF condition met, THEN assign A, ELSE assign B`). Vectorization allows this operation to be executed in compiled C code under the hood, maximizing efficiency over hundreds of thousands of rows compared to slow, iterative Python loops.